In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import scanpy as sc
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

In [ ]:
import spatialdata as sd

In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import squidpy as sq
import seaborn as sns
from shapely.geometry import Point, Polygon
from pathlib import Path
from typing import Dict, List, Tuple
import matplotlib.patches as patches

In [ ]:
import matplotlib
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt

In [ ]:
sc.set_figure_params(scanpy=True, fontsize=12)

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
import os
# --- PATH CONFIGURATION ---
# Base directory where the raw Xenium data is stored
data_dir = "/xenium"

# Base directory for outputs and temporary files (using scratch to avoid quota issues)
output_dir = "/xenium_output"
os.makedirs(output_dir, exist_ok=True)

# Set environment variables so temporary background files also write to scratch
scratch_tmp = os.path.join(output_dir, ".tmp")
os.makedirs(scratch_tmp, exist_ok=True)
os.environ["TMPDIR"] = scratch_tmp
os.environ["TEMP"] = scratch_tmp
os.environ["TMP"] = scratch_tmp

# Change working directory to the raw data directory for easy loading
os.chdir(data_dir)
print(f"Working directory set to: {os.getcwd()}")
print(f"Outputs will be saved to: {output_dir}")

In [ ]:
adata = sc.read_h5ad('annotated_data.h5ad') #import annotated data

In [ ]:
#====================================
#1. Metadata Preparation & Niche Construction
#====================================

In [ ]:
import squidpy as sq
import scanpy as sc

# --- SPATIAL NICHE IDENTIFICATION ---
# Initialize the column in the master adata with a placeholder
adata.obs['spatial_niche'] = "unassigned"

# Iterate through each unique tissue sample to calculate localized niches
for sample in adata.obs['tissue_id'].unique():
    print(f"Processing sample: {sample}...")
    
    # Isolate cells belonging to the current tissue slide
    mask = adata.obs['tissue_id'] == sample
    adata_sample = adata[mask].copy()
    
    # 1. Build a spatial graph restricted strictly to THIS slide
    sq.gr.spatial_neighbors(adata_sample, coord_type="generic", n_neighs=6)
    
    # 2. Cluster local environments using the slide's spatial connectivity matrix
    sc.tl.leiden(
        adata_sample, 
        adjacency=adata_sample.obsp["spatial_connectivities"], 
        key_added="temp_niche"
    )
    
    # Create globally unique niche identifiers (e.g., "SampleA_1")
    niche_labels = sample + "_" + adata_sample.obs['temp_niche'].astype(str)
    
    # 3. Map the slide-specific niche labels back to the master dataset
    adata.obs.loc[mask, 'spatial_niche'] = niche_labels

# --- VALIDATION AND QUALITY CONTROL ---
unassigned_count = (adata.obs['spatial_niche'] == "unassigned").sum()

In [ ]:
# --- CPDB INPUT PREPARATION & MASTER SAVE ---

# Create "Strict" niche-aware cell type names for CellPhoneDB (Format: CellType|SpatialNiche)
adata.obs['niche_aware_type'] = adata.obs['celltype2'].astype(str) + "|" + adata.obs['spatial_niche'].astype(str)

print("Sample of niche-aware annotations:")
print(adata.obs[['celltype2', 'spatial_niche', 'niche_aware_type']].head())

# Define the new safe path on your high-capacity scratch space
final_h5ad_path = os.path.join(output_dir, "spatial_with_niches_FINAL.h5ad")

In [ ]:
# --- CPDB MICROENVIRONMENTS MAPPING FILE ---
# 1. Isolate the unique combinations of niche-aware types and spatial niches
microenvs_final = adata.obs[['niche_aware_type', 'spatial_niche']].drop_duplicates()
# 2. Reset the index so the cell barcodes are removed from the mapping table
microenvs_final = microenvs_final.reset_index(drop=True)
# 3. Rename columns to match exact CellPhoneDB input requirements
microenvs_final.columns = ['cell_type', 'microenvironment']
# 4. Define safe saving path on scratch space to avoid quota limits
micro_output_path = os.path.join(output_dir, "microenvironments_spatial_strict.tsv")
# 5. Save the file without the row index
microenvs_final.to_csv(micro_output_path, sep='\t', index=False)

In [ ]:
import pandas as pd

# Extract cell identifiers and cell type annotations into a DataFrame
meta = pd.DataFrame({
    'Cell': adata.obs.index,
    'cell_type': adata.obs['niche_aware_type']  # Ensure 'celltype2' matches your annotation column exactly
})

# Define the output path in your scratch space
metadata_output_path = os.path.join(output_dir, "metadata.tsv")

# Save the metadata file as a tab-separated values (TSV) file
meta.to_csv(metadata_output_path, sep='\t', index=False)

print(f"Metadata successfully exported to: {metadata_output_path}")
print(f"Shape of metadata: {meta.shape}")

In [ ]:
#====================================
#2. Spatial niche manual check
#====================================

In [ ]:
# Plot the niches spatially, repeat for other tissues and conditions
sample_id = "s1432_neo"
adata_neo = adata[adata.obs['tissue_id'] == sample_id].copy()
sc.pl.spatial(adata_neo, color='spatial_niche', 
              title=f"Neonatal issue Architecture: {sample_id}", 
              spot_size=35, frameon=False, legend_loc=None)

# Compare with the actual cell types to see if niches = layers
sc.pl.spatial(adata_neo, color='celltype2', 
              title=f"Cell Types: {sample_id}", 
              spot_size=35, frameon=False)

In [ ]:
#====================================
#3. CellphoneDB run with microenvironment niches
#====================================

In [ ]:
import pandas as pd
import scanpy as sc
import os
import sys
import gc
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method

# --- PATH CONFIGURATION ---
scratch_dir = "/xenium_output"
os.makedirs(scratch_dir, exist_ok=True)

# Define exact locations of files created in previous steps
ADATA_PATH       = os.path.join(scratch_dir, "spatial_with_niches_FINAL.h5ad")
MICRO_MASTER_PATH = os.path.join(scratch_dir, "microenvironments_spatial_strict.tsv")

# The path to CellPhoneDB database file
CPDB_DB_PATH     = "/xenium/data/cellphonedb.zip" #change as needed

THREADS = 16
conditions = ['Fetal', 'Neonatal', 'NEC', 'Adult']

scratch_tmp = os.path.join(scratch_dir, ".tmp")
os.makedirs(scratch_tmp, exist_ok=True)
os.environ["TMPDIR"] = scratch_tmp
os.environ["TEMP"] = scratch_tmp
os.environ["TMP"] = scratch_tmp

# --- 1. LOAD MASTER DATA ---
print("Loading Master AnnData from scratch...")
adata = sc.read_h5ad(ADATA_PATH)
master_micro = pd.read_csv(MICRO_MASTER_PATH, sep='\t')

# --- 2. CONDITION ITERATION & RECOVERY ---
for cond in conditions:
    print(f"\n{'='*40}")
    print(f"STARTING RECOVERY ANALYSIS FOR: {cond}")
    print(f"{'='*40}")
    
    # Subset by condition
    adata_cond = adata[adata.obs['condition'] == cond].copy()
    
    # Fix potential duplicated cell barcode names within condition subsets
    adata_cond.obs_names_make_unique()
    
    # Drop any cell missing its niche assignment
    adata_cond = adata_cond[adata_cond.obs['niche_aware_type'].notna()].copy()

    # Filter Microenvironments to match only existing cell types in this subset
    cond_types = adata_cond.obs['niche_aware_type'].unique()
    micro_cond = master_micro[master_micro['cell_type'].isin(cond_types)].copy()
    
    # Establish distinct output directories per condition inside scratch
    out_path = os.path.join(scratch_dir, f"results_spatial_{cond}")
    os.makedirs(out_path, exist_ok=True)
    
    # Route all heavy temporary file pathways explicitly to scratch space
    temp_meta   = os.path.join(scratch_dir, f"meta_tmp_{cond}.tsv")
    temp_micro  = os.path.join(scratch_dir, f"micro_tmp_{cond}.tsv")
    temp_counts = os.path.join(scratch_dir, f"counts_tmp_{cond}.h5ad")
    
    # Save temporary subset slices to scratch
    meta = pd.DataFrame({
        'Cell': adata_cond.obs_names, 
        'cell_type': adata_cond.obs['niche_aware_type']
    })
    
    print(f"Writing temporary run files to scratch for {cond}...")
    meta.to_csv(temp_meta, sep='\t', index=False)
    micro_cond.to_csv(temp_micro, sep='\t', index=False)
    adata_cond.write_h5ad(temp_counts)

    # --- 3. RUN CELLPHONEDB ---
    try:
        print(f"🛰️ Calling CellPhoneDB statistical analysis for {cond}...")
        cpdb_statistical_analysis_method.call(
            cpdb_file_path = CPDB_DB_PATH, 
            meta_file_path = temp_meta, 
            counts_file_path = temp_counts, 
            microenvs_file_path = temp_micro, 
            counts_data = 'hgnc_symbol',
            output_path = out_path,
            threshold = 0.1,
            iterations = 1000,
            threads = THREADS,
            subsampling = False
        )
        print(f"✅ Successfully finished CellPhoneDB pipeline for: {cond}")
        print(f"Results located at: {out_path}")
    except Exception as e:
        print(f"❌ FAILED on condition {cond}: {e}")
    
    # --- 4. MEMORY & DISK CLEANUP ---
    print(f"Cleaning up temporary slice files for {cond}...")
    for f in [temp_meta, temp_micro, temp_counts]:
        if os.path.exists(f): 
            os.remove(f)
            
    del adata_cond
    gc.collect()

print("\n--- ALL RUNS FINISHED SUCCESSFULLY ---")